# 04.R4 — ResNet-50 Held-Out Test Evaluation

**Phase 04 — Document Processing & OCR (ResNet-50 Visual Pipeline)**  
**Objective:** Load `models/phase_04/resnet50/best_model.pt` and evaluate strictly on the untouched `test` partition (Accuracy, Macro-F1, Classification Report, Confusion Matrix, and 2048-d embedding intra-class vs. inter-class cosine similarity). Saves `models/phase_04/resnet50/evaluation_metrics.json`.


In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms

def _find_root(p: Path = Path.cwd()) -> Path:
    for parent in [p.resolve(), *p.resolve().parents]:
        if (parent / 'data').exists() and (parent / 'experiments').exists():
            return parent
    return p.resolve()

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.document_processing import DocumentProcessingPaths
paths = DocumentProcessingPaths(PROJECT_ROOT)

MANIFEST_CSV = paths.interim_doc_proc / 'resnet50' / 'resnet50_dataset_manifest.csv'
MODEL_DIR = paths.resnet50_model_dir
BEST_CKPT = MODEL_DIR / 'best_model.pt'
LABEL_MAP_JSON = MODEL_DIR / 'label_mapping.json'
EVAL_JSON = MODEL_DIR / 'evaluation_metrics.json'

if not BEST_CKPT.exists():
    raise FileNotFoundError(f"Best ResNet-50 checkpoint not found at {BEST_CKPT}. Run 03_resnet50_feature_extraction.ipynb first.")

df_test = pd.read_csv(MANIFEST_CSV)
df_test = df_test[df_test['split'] == 'test'].reset_index(drop=True)
label_info = json.loads(LABEL_MAP_JSON.read_text(encoding='utf-8'))
idx2label = {int(k): v for k, v in label_info['idx2label'].items()}
class_names = [idx2label[i] for i in range(label_info['num_classes'])]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class TestDocDS(Dataset):
    def __init__(self, df, root, tf):
        self.df, self.root, self.tf = df, root, tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        im = ImageOps.exif_transpose(Image.open(self.root / r['relative_path'])).convert('RGB')
        return self.tf(im), int(r['label_id'])

ckpt = torch.load(str(BEST_CKPT), map_location=DEVICE)
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, label_info['num_classes'])
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(DEVICE).eval()

# Hook penultimate 2048-d pooled embedding
embeddings_buf = []
def _hook(_m, inp, _out):
    embeddings_buf.append(inp[0].detach().cpu().numpy())
handle = model.fc.register_forward_hook(_hook)

y_true, y_pred = [], []
loader = DataLoader(TestDocDS(df_test, paths.root, eval_tf), batch_size=16, shuffle=False, num_workers=0)
with torch.no_grad():
    for imgs, lbls in loader:
        out = model(imgs.to(DEVICE))
        y_pred.extend(out.argmax(dim=1).cpu().numpy().tolist())
        y_true.extend(lbls.numpy().tolist())
handle.remove()

embs = np.vstack(embeddings_buf) if embeddings_buf else np.zeros((0, 2048))
acc = float(accuracy_score(y_true, y_pred))
macro_f1 = float(f1_score(y_true, y_pred, average='macro', zero_division=0))

# Intra-class vs inter-class cosine similarity
sim = cosine_similarity(embs) if len(embs) > 1 else np.zeros((1, 1))
intra_vals, inter_vals = [], []
for i in range(len(y_true)):
    for j in range(i + 1, len(y_true)):
        (intra_vals if y_true[i] == y_true[j] else inter_vals).append(float(sim[i, j]))

eval_metrics = {
    'evaluated_test_samples': len(y_true),
    'test_accuracy': round(acc, 4),
    'test_macro_f1': round(macro_f1, 4),
    'mean_intra_class_cosine_sim': round(float(np.mean(intra_vals)) if intra_vals else 0.0, 4),
    'mean_inter_class_cosine_sim': round(float(np.mean(inter_vals)) if inter_vals else 0.0, 4),
    'checkpoint_path': str(BEST_CKPT),
}
with open(EVAL_JSON, 'w', encoding='utf-8') as f:
    json.dump(eval_metrics, f, indent=2)

display(pd.DataFrame([eval_metrics]))
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap='Blues', xticks_rotation=35)
ax.set_title('ResNet-50 Held-Out Test Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()
